1. Initial Setup and Loading Data

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
import xgboost as xgb

# Load the datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (4209, 378)
Test shape: (4209, 377)


2. Check for Null and Unique Values

In [3]:
# Check for null values
print("Any missing values in train set?", train_df.isnull().sum().any())
print("Any missing values in test set?", test_df.isnull().sum().any())

# Check unique values for each column
print("\nUnique values in train set (first 10 columns):")
print(train_df.nunique().head(10))

Any missing values in train set? False
Any missing values in test set? False

Unique values in train set (first 10 columns):
ID    4209
y     2545
X0      47
X1      27
X2      44
X3       7
X4       4
X5      29
X6      12
X8      25
dtype: int64


3. Remove Columns with Zero Variance

In [4]:
# Calculate variance of numerical columns only
variances = train_df.var(numeric_only=True)

# Find columns where variance is exactly 0
zero_variance_cols = variances[variances == 0].index.tolist()
print(f"Dropping {len(zero_variance_cols)} columns with zero variance: {zero_variance_cols}")

# Drop them from BOTH train and test sets
train_df.drop(columns=zero_variance_cols, inplace=True)
test_df.drop(columns=zero_variance_cols, inplace=True)

Dropping 12 columns with zero variance: ['X11', 'X93', 'X107', 'X233', 'X235', 'X268', 'X289', 'X290', 'X293', 'X297', 'X330', 'X347']


4. Apply Label Encoder

In [5]:
# Identify categorical columns (type 'object')
categorical_cols = train_df.select_dtypes(include=['object']).columns

# Initialize LabelEncoder
le = LabelEncoder()

for col in categorical_cols:
    # We fit on the combined data of train and test to ensure the encoder
    # recognizes all possible unique labels across both datasets
    combined_data = list(train_df[col].values) + list(test_df[col].values)
    le.fit(combined_data)

    # Transform the columns
    train_df[col] = le.transform(list(train_df[col].values))
    test_df[col] = le.transform(list(test_df[col].values))

print("Categorical columns successfully encoded.")

Categorical columns successfully encoded.


5. Perform Dimensionality Reduction (PCA)

In [6]:
# Separate features (X) from target (y) and drop ID
X_train = train_df.drop(['ID', 'y'], axis=1)
y_train = train_df['y']
X_test = test_df.drop(['ID'], axis=1)

# Initialize PCA

pca = PCA(n_components=12, random_state=42)

# Fit and transform the training data, just transform the test data
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Original feature count: {X_train.shape[1]}")
print(f"Reduced feature count (PCA): {X_train_pca.shape[1]}")

Original feature count: 364
Reduced feature count (PCA): 12


6. Predict test_df Values Using XGBoost

In [7]:
# Initialize the XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    max_depth=4,
    learning_rate=0.1,
    n_estimators=150,
    random_state=42
)

# Train the model
xgb_model.fit(X_train_pca, y_train)
print("Model training complete.")

# Predict on the test set
test_predictions = xgb_model.predict(X_test_pca)

#final dataframe
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'y': test_predictions
})

# Save to CSV
submission.to_csv('mercedes_xgboost_submission.csv', index=False)
print("Predictions saved to 'mercedes_xgboost_submission.csv'.")

Model training complete.
Predictions saved to 'mercedes_xgboost_submission.csv'.
